# Major and Minor Player with Partial Observations

In this notebook, we compare the learned and exact equilibria of the major-minor linear-quadratic game of Firoozi
and Caines (2020), with one minor player.

In [1]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import MajorMinor

device: cpu


## 1. The game

We denote by $x^0$ the state of the major player and by $x^1$ the state of the minor player. Each player controls
its state, and the major player's state enters the drift of the minor player's state. The states follow

$$
\begin{aligned}
dx^0_t &= (A_0\, x^0_t + B_0\, u^0_t)\,dt + D_0\,dw^0_t, \\[4pt]
dx^1_t &= (A\, x^1_t + G\, x^0_t + B\, u^1_t)\,dt + D\,dw^1_t ,
\end{aligned}
$$

where $w^0$ and $w^1$ are independent Brownian motions. Neither player observes its state directly. The major player
observes $y^0$, a noisy measurement of its own state. The minor player observes $y^1$, noisy measurements of its own
state and of the major player's state. The observations follow

$$
\begin{aligned}
dy^0_t &= L_0\, x^0_t\,dt + \sigma_{v_0}\,dv^0_t, \\[4pt]
dy^1_t &= (L_1\, x^1_t,\; L_2\, x^0_t)\,dt + \sigma_v\,dv^1_t ,
\end{aligned}
$$

where $v^0$ and $v^1$ are Brownian motions independent of $w^0$ and $w^1$, and $v^1$ is two-dimensional. Neither
player observes the other's observations, so neither player's information contains the other's. The players
minimize the discounted costs

$$
\begin{aligned}
J^0 &= \mathbb{E} \int_0^T e^{-\rho t} \big[ Q_0\,(x^0_t - \eta_0)^2 + R_0\,(u^0_t)^2 \big]\,dt, \\[4pt]
J^1 &= \mathbb{E} \int_0^T e^{-\rho t} \big[ Q\,(x^1_t - H_1 x^0_t - \eta)^2 + R\,(u^1_t)^2 \big]\,dt .
\end{aligned}
$$

Thus the major player keeps its state close to $\eta_0$, and the minor player keeps its state close to
$H_1 x^0 + \eta$.

The game differs from the paper in three ways. There is one minor player instead of many minor players coupled
through their average, so the cost terms that depend on that average are zero. The horizon is finite, $T = 5$, with
discounting. The states are scalar, and the coefficients $L_0$, $L_1$ and $L_2$ replace the observation matrices of
the paper.

## 2. Exact equilibrium

The game is linear-quadratic and Gaussian. Thus the control of each player is linear in its estimate of the
quantities that enter its problem. We denote by $\hat x^0$ the Kalman filter of $x^0$ given $y^0$. The cost of the
major player depends only on its own state, so the major player's control is linear in $\hat x^0$.

The minor player needs $x^1$ and $x^0$. The major player's control depends on $\hat x^0$, so the future of $x^0$
also depends on $\hat x^0$. Thus the minor player filters $(x^1, x^0, \hat x^0)$ given $y^1$, and the minor player's
control is linear in this filter. The minor player estimates the major player's estimate.

`MajorMinor.exact` computes both filters by the discrete Kalman filter and both controls by a discounted backward
Riccati recursion, on the discretised game. The learned method uses none of these. The network of each player takes
only its own observations as input.

## 3. Learned and exact equilibria

The learned method is described in the README, here with 5,000 paths, damping $\rho = 0.7$ and 6 iterations. The
figure compares the learned and exact controls on three validation paths.

In [ ]:
game = MajorMinor()
solver = Solver(game, damping=0.7, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(6, average_from=3, reference=exact)
print(f'relative L2 error: major {relative_l2(learned[0], exact[0]):.3f}, minor {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['major player', 'minor player'])

iteration  0  loss 3.81e-02  error 0.278 0.325  (94s)
